<a href="https://colab.research.google.com/github/BillJr99/Ursinus-CS357/blob/gh-pages/files/notebooks/Sentence_Prediction_with_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentence Prediction with BERT

This notebook demonstrates how to use a masked‑language‑model (BERT) to **predict the next word at the end of a sentence** by inserting a special `<mask>` token and asking the model to fill it in. The code is kept minimal and uses Hugging Face `transformers` with PyTorch.


## 1. Install Dependencies

We install two packages:
- **`sentencepiece`** — some tokenizers rely on it.
- **`transformers`** — Hugging Face library providing pretrained BERT and utilities.

In Colab, `pip install` within a notebook cell updates the current runtime only.


In [ ]:
"""
Demo: Sentence completion
https://ajay-arunachalam08.medium.com/an-illustration-of-next-word-prediction-with-state-of-the-art-network-architectures-like-bert-gpt-c0af02921f17
"""

!pip install sentencepiece
!pip install transformers

## Imports and Logging

We import PyTorch and Hugging Face classes for **BERT masked language modeling**. We also quiet nonessential logs to keep outputs focused on results.


In [ ]:
import torch
from torch.nn import functional as F
import string
from transformers import BertTokenizer, BertForMaskedLM, logging
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

logging.set_verbosity_error()

## Placeholders for Configuration

We declare three global placeholders that will later receive values returned by `set_model_config`. They are used to control:
- `no_words_to_be_predicted`: how many candidate words to retrieve
- `select_model`: which model family to use (here, just `'bert'`)
- `enter_input_text`: the user‑provided sentence prefix.


In [ ]:
# declare variables
no_words_to_be_predicted = globals()
select_model = globals()
enter_input_text = globals()

In [ ]:
# set model configuration
def set_model_config(**kwargs):
  for key, value in kwargs.items():
    print("{0} = {1}".format(key, value))

  no_words_to_be_predicted = list(kwargs.values())[0]  # integer values
  select_model = list(kwargs.values())[1]  # possible values = 'bert'
  enter_input_text = list(kwargs.values())[2]  #only string

  return no_words_to_be_predicted, select_model, enter_input_text

## Load Pretrained BERT

`load_model(model_name)` fetches the **tokenizer** and the **masked‑LM head** for `bert-base-uncased` and switches the model to `.eval()` for inference. We return both objects for downstream use.


In [ ]:
# load model and tokenizer
def load_model(model_name):
  try:
    if model_name.lower() == "bert":
      bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
      bert_model = BertForMaskedLM.from_pretrained('bert-base-uncased').eval()
      return bert_tokenizer, bert_model
  except Exception as e:
    pass


## Encode/Decode Utilities and Prediction

**Encoding (`encode_bert`)**  

- Replaces any literal `<mask>` with the tokenizer’s true mask token (typically `[MASK]`).
- If the mask is the last token, appends a period so the model tends not to predict punctuation.
- Returns `input_ids` and the `mask_idx` position.**Decoding (`decode_bert`)**
- Converts predicted token IDs back to strings.
- Removes punctuation and WordPiece continuation markers (`##`).
- Returns the top-*k* tokens joined by newlines.**Prediction helpers**
- `get_all_predictions` runs the model, selects the top `no_words_to_be_predicted` candidates at the masked position, and decodes them.
- `get_prediction_end_of_sentence` appends a `<mask>` to the user input and calls `get_all_predictions`.

This illustrates **masked language modeling** (MLM). Although BERT is not a generative autoregressive model, we can emulate “next‑word” prediction by asking it to fill a single `<mask>` placed at the end of the input.


In [ ]:
# bert encode
def encode_bert(tokenizer, text_sentence, add_special_tokens=True):
  text_sentence = text_sentence.replace('<mask>', tokenizer.mask_token)
  # if <mask> is the last token, append a "." so that models dont predict punctuation.
  if tokenizer.mask_token == text_sentence.split()[-1]:
    text_sentence += ' .'
    input_ids = torch.tensor([
        tokenizer.encode(text_sentence, add_special_tokens=add_special_tokens)
    ])
    mask_idx = torch.where(input_ids == tokenizer.mask_token_id)[1].tolist()[0]
  return input_ids, mask_idx

# bert decode
def decode_bert(tokenizer, pred_idx, top_clean):
  ignore_tokens = string.punctuation + '[PAD]'
  tokens = []
  for w in pred_idx:
    token = ''.join(tokenizer.decode(w).split())
    if token not in ignore_tokens:
      tokens.append(token.replace('##', ''))
  return '\n'.join(tokens[:top_clean])

# Get the probability associated with each predicted token
def get_all_probabilities(predict, mask_idx, top_clean=5):
  # get logits for the masked position
  logits = predict[0, mask_idx, :]

  # convert to probabilities
  probs = F.softmax(logits, dim=-1)

  # select top-k indices and their probabilities
  topk = torch.topk(probs, no_words_to_be_predicted)
  indices = topk.indices.tolist()
  values = topk.values.tolist()

  # slice probabilities in parallel to top_clean
  bert_probs = values[:top_clean]

  return bert_probs

def get_all_predictions(text_sentence, model_name, top_clean=5):
  #print(model_name)
  if model_name.lower() == "bert":
    # ========================= BERT =================================
    input_ids, mask_idx = encode_bert(bert_tokenizer, text_sentence)
    with torch.no_grad():
      predict = bert_model(input_ids)[0]
    bert = decode_bert(
        bert_tokenizer,
        predict[0,
                mask_idx, :].topk(no_words_to_be_predicted).indices.tolist(),
        top_clean)
    probs = get_all_probabilities(predict, mask_idx, top_clean)
    return {'bert': bert, 'probabilities': probs}

def get_prediction_end_of_sentence(input_text, model_name):
  try:
    if model_name.lower() == "bert":
      input_text += ' <mask>'
      print(input_text)
      res = get_all_predictions(input_text,
                                model_name,
                                top_clean=int(no_words_to_be_predicted))
      return res
  except Exception as error:
    pass


## Try it Out

The final cell:

1. Prompts for a sentence beginning (e.g., `Once upon a`).
2. Sets the configuration (top‑`k` = 5 by default).
3. Loads BERT and obtains predictions for the word at `<mask>`.
4. Prints the candidates in a compact list.
  
Example Input: `Once upon a`  Model input becomes: `Once upon a <mask>`  

Output could be something like: `time  mattress  dream  lifetime  kiss` (your results may vary due to model stochasticity).


In [ ]:
try:
  print("Next Word Prediction with Pytorch using BERT")
  input_string = input(
      "Enter the beginning of a sentence to complete:")  # "why are"
  selected_model = "bert"
  no_words_to_be_predicted, select_model, enter_input_text = set_model_config(
      no_words_to_be_predicted=5,
      select_model=selected_model,
      enter_input_text=input_string)
  if select_model.lower() == "bert":
    bert_tokenizer, bert_model = load_model(select_model)
    res = get_prediction_end_of_sentence(enter_input_text, select_model)
    print("Predictions with probabilities:")
    for token, p in zip(res['bert'].split("\n"), res['probabilities']):
        print(f"{token:15s} {p*100:6.2f}%")

except Exception as e:
  print('Some problem occured', e)

Next Word Prediction with Pytorch using BERT
Enter the beginning of a sentence to complete:Once upon a
no_words_to_be_predicted = 5
select_model = bert
enter_input_text = Once upon a


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Once upon a <mask>
Predictions with probabilities:
time             99.28%
mattress          0.39%
dream             0.02%
lifetime          0.01%
kiss              0.01%
